![](Images/banner_health_analytics.png){fig-align="center" width="100%"}

### Health Analytics with Python · Module 04: Machine Learning for Clinical Prediction
***
**Learning objectives**
- Understand Shapley values and the SHAP framework conceptually
- Generate global explanations (summary, bar, beeswarm plots)
- Produce local explanations (waterfall, force plots) for individual patients
- Build SHAP dependence plots to reveal interaction effects
- Create a clinical explanation report for a high-risk patient

**Estimated time:** 2.5 hours | **Level:** Advanced | **Libraries:** `shap`, `sklearn`, `matplotlib`


## 1. Setup & Train Model

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
import warnings; warnings.filterwarnings("ignore")
import os; os.makedirs("/tmp/mod04",exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})

try:
    import shap
    print(f"SHAP {shap.__version__} available")
except ImportError:
    print("Run: pip install shap")

np.random.seed(42); N=3000
def sig(x): return 1/(1+np.exp(-x))
b=np.random.normal(size=(N,4)); np.random.seed(99)
age=np.random.normal(62,15,N).clip(18,92).astype(int)
sex=np.random.choice(["M","F"],N,p=[0.48,0.52])
payer=np.random.choice(["Medicare","Medicaid","Private","Self-pay","Other"],N,p=[0.40,0.22,0.28,0.07,0.03])
admit_type=np.random.choice(["Emergency","Elective","Urgent"],N,p=[0.52,0.30,0.18])
los_days=np.random.gamma(2.5,1.8,N).clip(1,30).astype(int)
diabetes=np.random.binomial(1,sig(0.6*b[:,0]-0.5),N)
hypertension=np.random.binomial(1,sig(0.7*b[:,0]-0.2),N)
hf=np.random.binomial(1,sig(0.4*b[:,1]+0.5*hypertension-1.5),N)
copd=np.random.binomial(1,sig(0.5*b[:,2]-1.0),N)
ckd=np.random.binomial(1,sig(0.5*b[:,0]+0.6*diabetes-1.2),N)
obesity=np.random.binomial(1,sig(0.5*b[:,0]-0.8),N)
depression=np.random.binomial(1,sig(0.3*b[:,3]-1.4),N)
n_comorb=diabetes+hypertension+hf+copd+ckd
np.random.seed(21)
glucose=np.random.normal(105+15*diabetes,28,N).clip(50,400).round(1)
creatinine=np.random.gamma(1.6+0.8*hf,0.6,N).clip(0.4,10).round(2)
hba1c=np.random.normal(6.5+0.8*diabetes,1.4,N).clip(4,14).round(1)
sbp=np.random.normal(128+12*hypertension,18,N).clip(80,200).round(0)
bmi=np.random.normal(28+4*obesity,6,N).clip(15,55).round(1)
logit=(-3.2+0.6*hf+0.5*diabetes+0.5*ckd+0.3*copd+0.02*(age-62)/15
       +0.3*(admit_type=="Emergency").astype(int)+0.25*(payer=="Medicaid").astype(int)
       +0.2*(los_days>7).astype(int)+0.5*hf*diabetes+np.random.normal(0,0.25,N))
readmitted=np.random.binomial(1,sig(logit),N)
df=pd.DataFrame({
    "age":age,"sex":sex,"payer":payer,"admit_type":admit_type,"los_days":los_days,
    "diabetes":diabetes,"hypertension":hypertension,"hf":hf,"copd":copd,"ckd":ckd,
    "obesity":obesity,"depression":depression,"n_comorb":n_comorb,
    "glucose":glucose,"creatinine":creatinine,"hba1c":hba1c,"sbp":sbp,"bmi":bmi,"readmitted":readmitted,
})
df["los_gt7"]=(df.los_days>7).astype(int)
NUMERIC=["age","los_days","n_comorb","glucose","creatinine","hba1c","sbp","bmi"]
BINARY=["diabetes","hypertension","hf","copd","ckd","obesity","depression","los_gt7"]
CATEGORIC=["sex","payer","admit_type"]
FEATURES=NUMERIC+BINARY+CATEGORIC; TARGET="readmitted"
pre=ColumnTransformer([
    ("num",Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler())]),NUMERIC),
    ("bin",SimpleImputer(strategy="most_frequent"),BINARY),
    ("cat",Pipeline([("imp",SimpleImputer(strategy="most_frequent")),
                     ("ohe",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]),CATEGORIC),
])
X=df[FEATURES]; y=df[TARGET]
X_tr,X_te,y_tr,y_te=train_test_split(X,y,test_size=0.20,stratify=y,random_state=42)
pre.fit(X_tr); Xtr=pre.transform(X_tr); Xte=pre.transform(X_te)
ohe_nm=pre.named_transformers_["cat"]["ohe"].get_feature_names_out(CATEGORIC)
feat_nm=NUMERIC+BINARY+list(ohe_nm)
X_te_named=pd.DataFrame(Xte,columns=feat_nm)
X_tr_named=pd.DataFrame(Xtr,columns=feat_nm)

model=GradientBoostingClassifier(n_estimators=200,max_depth=4,learning_rate=0.05,
                                   subsample=0.8,random_state=42)
model.fit(Xtr,y_tr)
print(f"Model AUC: {roc_auc_score(y_te,model.predict_proba(Xte)[:,1]):.4f}")


## 2. SHAP — Conceptual Foundation

> **SHAP value** for feature $j$ on observation $i$: the average marginal contribution of feature $j$ to the prediction, weighted over all possible subsets of features (from game theory's Shapley values).

Properties:
- **Additivity**: sum of all SHAP values = prediction − base rate
- **Consistency**: if a feature contributes more in model A than B, its SHAP value is larger in A
- **Local accuracy**: SHAP values exactly decompose each individual prediction


In [ ]:
# ── Compute SHAP values ───────────────────────────────────────────────────────
try:
    explainer = shap.TreeExplainer(model)
    shap_vals = explainer.shap_values(X_te_named)   # shape: (n_samples, n_features)
    shap_vals_sample = shap_vals[:300]  # use 300 for speed in plots
    X_sample = X_te_named.iloc[:300]

    print(f"SHAP values shape: {shap_vals.shape}")
    print(f"Expected value (base rate): {explainer.expected_value:.4f}")
    print(f"  (corresponds to: {sig(explainer.expected_value)*100:.1f}% readmission probability)")

    # Verify additivity: prediction = base + sum(SHAP)
    idx=0
    pred_prob = model.predict_proba(Xte[idx:idx+1])[0,1]
    shap_sum  = explainer.expected_value + shap_vals[idx].sum()
    print(f" Additivity check (patient 0):")
    print(f"  Model prediction    : {pred_prob:.4f}")
    print(f"  Base + sum(SHAP)    : {sig(shap_sum):.4f}")
except Exception as e:
    print(f"SHAP error: {e}"); shap_vals=None


In [ ]:
# ── Global summary plot ───────────────────────────────────────────────────────
if shap_vals is not None:
    fig, axes = plt.subplots(1,2,figsize=(16,7))

    # Bar plot: mean |SHAP| per feature (global importance)
    mean_abs = np.abs(shap_vals).mean(axis=0)
    top_idx  = np.argsort(mean_abs)[-15:]
    axes[0].barh([feat_nm[i] for i in top_idx], mean_abs[top_idx],
                  color="#4878CF", alpha=0.85, edgecolor="white")
    axes[0].set_xlabel("Mean |SHAP value|")
    axes[0].set_title("Global Feature Importance (mean |SHAP value| = average impact on model output)")

    # Beeswarm plot (manual version)
    top15_idx = np.argsort(mean_abs)[-15:]
    for rank, feat_idx in enumerate(top15_idx):
        shap_col = shap_vals_sample[:, feat_idx]
        feat_col = X_sample.iloc[:, feat_idx]
        # Normalise feature value for color
        norm_val = (feat_col - feat_col.min()) / (feat_col.max() - feat_col.min() + 1e-8)
        sc = axes[1].scatter(shap_col, np.full_like(shap_col, rank) + np.random.normal(0,0.08,len(shap_col)),
                             c=norm_val, cmap="RdBu_r", alpha=0.4, s=6)
    axes[1].set_yticks(range(len(top15_idx)))
    axes[1].set_yticklabels([feat_nm[i] for i in top15_idx], fontsize=9)
    axes[1].axvline(0, color="black", lw=0.8)
    axes[1].set_xlabel("SHAP value (impact on log-odds)")
    axes[1].set_title("Beeswarm Plot (red=high feature value, blue=low)")
    plt.colorbar(sc, ax=axes[1], label="Feature value (normalised)")
    plt.tight_layout()
    plt.savefig("/tmp/mod04/shap_global.png", bbox_inches="tight"); plt.show()


In [ ]:
# ── Local explanation: high-risk patient waterfall ────────────────────────────
if shap_vals is not None:
    # Find a high-risk patient
    probs_te = model.predict_proba(Xte)[:,1]
    high_risk_idx = np.argsort(probs_te)[-1]   # highest predicted probability

    shap_patient = shap_vals[high_risk_idx]
    feat_vals    = X_te_named.iloc[high_risk_idx]
    pred_patient = probs_te[high_risk_idx]

    # Manual waterfall plot
    top_k = 12
    top_local_idx = np.argsort(np.abs(shap_patient))[-top_k:][::-1]
    labels = [f"{feat_nm[i]} = {feat_vals.iloc[i]:.2f}" for i in top_local_idx]
    vals   = shap_patient[top_local_idx]
    colors = ["#D65F5F" if v > 0 else "#4878CF" for v in vals]

    fig, ax = plt.subplots(figsize=(9,6))
    ax.barh(range(len(vals)), vals[::-1], color=colors[::-1], edgecolor="white", alpha=0.85)
    ax.set_yticks(range(len(vals))); ax.set_yticklabels(labels[::-1], fontsize=9)
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("SHAP value (impact on readmission log-odds)")
    ax.set_title(f"Local Explanation — High-risk Patient Predicted readmission probability: {pred_patient:.1%} Actual outcome: {'Readmitted' if y_te.iloc[high_risk_idx]==1 else 'Not readmitted'}")
    for i,(v,lbl) in enumerate(zip(vals[::-1],labels[::-1])):
        direction = "→ increases risk" if v>0 else "→ decreases risk"
        ax.text(v + (0.002 if v>0 else -0.002), i, f"{v:+.3f} {direction}",
                va="center", ha="left" if v>0 else "right", fontsize=7)
    plt.tight_layout(); plt.savefig("/tmp/mod04/shap_waterfall.png", bbox_inches="tight"); plt.show()


In [ ]:
# ── SHAP dependence plot: hf × diabetes interaction ──────────────────────────
if shap_vals is not None:
    hf_idx  = feat_nm.index("hf")
    dm_idx  = feat_nm.index("diabetes")
    fig, axes = plt.subplots(1,2,figsize=(14,5))

    for ax, main_idx, color_idx, main_name, color_name in [
        (axes[0], hf_idx,  dm_idx,  "hf",        "diabetes (0/1)"),
        (axes[1], dm_idx,  hf_idx,  "diabetes",  "hf (0/1)"),
    ]:
        sc = ax.scatter(
            X_sample.iloc[:,main_idx],
            shap_vals_sample[:,main_idx],
            c=X_sample.iloc[:,color_idx],
            cmap="RdBu_r", alpha=0.5, s=15
        )
        plt.colorbar(sc, ax=ax, label=color_name)
        ax.axhline(0, color="black", lw=0.8, ls="--")
        ax.set_xlabel(main_name); ax.set_ylabel(f"SHAP value for {main_name}")
        ax.set_title(f"Dependence plot: {main_name} (colour = {color_name})")

    plt.tight_layout(); plt.savefig("/tmp/mod04/shap_dependence.png", bbox_inches="tight"); plt.show()
    print("Dependence plot interpretation:")
    print("  Vertical spread at a given x-value = interaction with the colour variable")
    print("  If red points (diabetes=1) cluster higher, HF×DM interaction drives risk")


## 3. Clinical Report Template
```
=== READMISSION RISK EXPLANATION REPORT ===
Patient ID : PT-XXXXX
Admit date : 2023-05-12
Predicted 30-day readmission risk: 38.4% (HIGH RISK — above 20% threshold)
Population baseline risk: 14.2%

TOP RISK-INCREASING FACTORS:
  1. Heart failure present        +0.41  (strong positive contribution)
  2. Creatinine = 2.8 mg/dL      +0.29  (elevated kidney function marker)
  3. LOS = 12 days                +0.18  (extended stay)
  4. Diabetes present             +0.14
  5. HbA1c = 9.2%                 +0.12  (poor glycaemic control)

TOP RISK-DECREASING FACTORS:
  1. Private insurance            -0.08  (resource access)
  2. Age = 54 years               -0.05  (younger than average)

RECOMMENDED ACTIONS (per clinical pathway):
  - Enrol in heart failure management programme
  - Nephrology follow-up within 7 days
  - Pharmacy medication reconciliation at discharge
===========================================
```


## 4. Knowledge Check
1. A SHAP value of +0.35 for `hf`. What does this mean in terms of log-odds and probability?
2. The beeswarm plot shows `creatinine` has a wide spread of SHAP values. What does this imply about its clinical importance?
3. A patient has SHAP sum = +0.8 but was NOT readmitted. Is the model wrong?
4. When should you use `shap.KernelExplainer` instead of `shap.TreeExplainer`?
5. Build a SHAP interaction heatmap showing the top 6×6 pairwise interactions.


In [ ]:
# Exercise 5 — SHAP interaction heatmap
if shap_vals is not None:
    top6_idx = np.argsort(np.abs(shap_vals).mean(axis=0))[-6:]
    top6_nm  = [feat_nm[i] for i in top6_idx]
    corr_mat = np.corrcoef(shap_vals[:,top6_idx].T)
    fig,ax=plt.subplots(figsize=(7,6))
    sns.heatmap(pd.DataFrame(corr_mat,index=top6_nm,columns=top6_nm),
                cmap="RdBu_r",center=0,annot=True,fmt=".2f",linewidths=0.5,ax=ax)
    ax.set_title("SHAP value correlation (top 6 features)")
    plt.tight_layout(); plt.show()


***
**Next → NB-07: Three Clinical Prediction Models (Sepsis, ICU LOS, Readmission)**
